# Kotoba TTS - Real-time Text-to-Speech with Bidirectional Streaming

This notebook demonstrates how to use the Kotoba TTS (Text-to-Speech) model package from AWS Marketplace.

**Product**: Kotoba TTS - Real-time Japanese Speech Synthesis  
**Seller**: Kotoba AI

## Key Features

- Real-time streaming synthesis with sub-second time-to-first-audio
- Native Japanese voices plus English; custom speaker embeddings supported
- Multiple audio codecs (`pcm_f32`, `pcm16`, `float32`)
- Scalable GPU-accelerated inference

---

## IMPORTANT: Bidirectional Streaming Only

This product uses **SageMaker bidirectional streaming** exclusively:

- Standard `POST /invocations` is **NOT supported**
- Batch transform jobs are **NOT supported**
- Requires **Python 3.12+** with `aws-sdk-python[sagemaker-runtime-http2]`

---

## Prerequisites

1. **AWS Account** with SageMaker permissions
2. **Python 3.12+** (required for HTTP/2 SDK)
3. **AWS Marketplace subscription** to Kotoba TTS

### Required IAM Permissions

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "sagemaker:CreateEndpoint",
                "sagemaker:CreateEndpointConfig",
                "sagemaker:DeleteEndpoint",
                "sagemaker:DeleteEndpointConfig",
                "sagemaker:DescribeEndpoint",
                "sagemaker:InvokeEndpoint",
                "sagemaker:InvokeEndpointWithResponseStream"
            ],
            "Resource": "*"
        }
    ]
}
```

## Table of Contents

1. [Subscribe to Model Package](#1.-Subscribe-to-Model-Package)
2. [Setup](#2.-Setup)
3. [Deploy Real-time Endpoint](#3.-Deploy-Real-time-Endpoint)
4. [Perform Real-time Synthesis](#4.-Perform-Real-time-Synthesis)
5. [Clean-up](#5.-Clean-up)
6. [Appendix A: AWS Marketplace Listing Fields](#Appendix-A:-AWS-Marketplace-Listing-Fields)

## 1. Subscribe to Model Package

To subscribe to the Kotoba TTS model package:

1. Open the [Kotoba TTS listing on AWS Marketplace](#) <!-- TODO: Update with actual URL -->
2. Click **Continue to Subscribe**
3. Review the pricing and EULA, then click **Subscribe**
4. Once subscribed, click **Continue to Configuration**
5. Select your preferred region and note the **Model Package ARN**

## 2. Setup

### 2.1 Check Python Version

In [ ]:
import sys

if sys.version_info < (3, 12):
    raise RuntimeError(
        f"Python 3.12+ is required for SageMaker bidirectional streaming. "
        f"Current version: {sys.version}"
    )

print(f"Python version: {sys.version}")

### 2.2 Install Dependencies

In [ ]:
!pip install -q boto3 sagemaker numpy
!pip install -q "aws-sdk-python[sagemaker-runtime-http2]"

### 2.3 Import Libraries

In [ ]:
import asyncio
import base64
import json
import time
import wave
from io import BytesIO

import boto3
import numpy as np
import sagemaker
from sagemaker import ModelPackage

# SageMaker HTTP/2 SDK for bidirectional streaming
from aws_sdk_sagemaker_runtime_http2.client import SageMakerRuntimeHTTP2Client
from aws_sdk_sagemaker_runtime_http2.config import Config, HTTPAuthSchemeResolver
from aws_sdk_sagemaker_runtime_http2.models import (
    InvokeEndpointWithBidirectionalStreamInput,
    RequestPayloadPart,
    RequestStreamEventPayloadPart,
)
from smithy_aws_core.auth.sigv4 import SigV4AuthScheme
from smithy_aws_core.identity import StaticCredentialsResolver

print("All dependencies imported successfully.")

### 2.4 Configure AWS Session

In [ ]:
# Initialize SageMaker session
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session.boto_region_name

print(f"Region: {region}")
print(f"Role: {role}")

### 2.5 Configure Model Package ARN

Replace `<MODEL_PACKAGE_ARN>` with the ARN from your AWS Marketplace subscription.

In [ ]:
# TODO: Replace with your Model Package ARN from AWS Marketplace
MODEL_PACKAGE_ARN = "<MODEL_PACKAGE_ARN>"

# Validate ARN format
if MODEL_PACKAGE_ARN.startswith("<"):
    raise ValueError(
        "Please replace MODEL_PACKAGE_ARN with your actual Model Package ARN "
        "from AWS Marketplace subscription."
    )

## 3. Deploy Real-time Endpoint

### 3.1 Create Endpoint

In [ ]:
# Endpoint configuration
ENDPOINT_NAME = f"kotoba-tts-{int(time.time())}"
INSTANCE_TYPE = "ml.g6.2xlarge"  # GPU instance for real-time inference

print(f"Endpoint name: {ENDPOINT_NAME}")
print(f"Instance type: {INSTANCE_TYPE}")

In [ ]:
# Create Model from Model Package
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=sagemaker_session,
)

# Deploy endpoint
print(f"Deploying endpoint '{ENDPOINT_NAME}'...")
print("This may take 5-10 minutes.")

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)

print(f"Endpoint '{ENDPOINT_NAME}' is now InService.")

## 4. Perform Real-time Synthesis

### 4.1 Connection Protocol

Kotoba TTS uses SageMaker bidirectional streaming:

```
Client (HTTP/2 SDK) -> SageMaker Runtime (HTTPS:8443) -> Sidecar -> Container (WS:8080)
```

**Note**: Standard WebSocket libraries cannot connect directly to SageMaker endpoints.

### 4.2 Output Audio Format Specifications

| Format    | Sample Rate | Channels | Encoding              |
| --------- | ----------- | -------- | --------------------- |
| `pcm_f32` | 24000 Hz    | 1 (mono) | Little-endian float32 |
| `pcm16`   | 24000 Hz    | 1 (mono) | Little-endian int16   |
| `float32` | 24000 Hz    | 1 (mono) | Little-endian float32 |
| `twilio`  | 8000 Hz     | 1 (mono) | μ-law                 |

The default output for this notebook is `pcm_f32` (what the server emits). Audio chunks are base64-encoded.

### 4.3 Protocol Flow

```
Client                                    Server
  |<---- session.created -------------------|
  |---- open ------------------------------>|
  |---- response.create ------------------->|
  |<---- response.created ------------------|
  |---- input_text_buffer.append ---------->| (concurrent)
  |<---- audio.chunk (isFinal:false) -------|
  |---- input_text_buffer.commit ---------->|
  |<---- audio.chunk (isFinal:true) --------|
  |<---- response.done ---------------------|
```

### 4.4 Event Reference

#### Client Events (you send)

| Event                         | Description                                             |
| ----------------------------- | ------------------------------------------------------- |
| `open`                        | First message; configure language / speaker_id          |
| `response.create`             | Start a synthesis turn (repeatable per session)         |
| `input_text_buffer.append`    | Send a text chunk (sentence-level recommended)          |
| `input_text_buffer.commit`    | Signal end of text input for this turn                  |
| `response.cancel`             | Cancel the currently active synthesis                   |

#### Server Events (you receive)

| Event                         | Description                                             |
| ----------------------------- | ------------------------------------------------------- |
| `session.created`             | Session established (format, sample_rate, channels)     |
| `response.created`            | Synthesis turn started                                  |
| `audio.chunk`                 | Base64 audio delta (`isFinal` marks the last chunk)     |
| `response.done`               | Turn finished (`status`: completed / cancelled / failed)|
| `error`                       | Fatal error (connection will close)                     |
| `timeout`                     | Worker result timeout notification                      |

### 4.5 Prepare Input Text

In [ ]:
SAMPLE_RATE = 24000
LANGUAGE = "ja"
SPEAKER_ID = "default"

# Input text to synthesize (split into sentence-level chunks for streaming).
TEXT_CHUNKS = [
    "こんにちは、",
    "世界。",
    "今日もよろしくお願いします。",
]

full_text = "".join(TEXT_CHUNKS)
print(f"Language: {LANGUAGE}")
print(f"Speaker: {SPEAKER_ID}")
print(f"Text: {full_text!r}")
print(f"Chunks: {len(TEXT_CHUNKS)}")


### 4.6 Streaming Inference

In [ ]:
async def synthesize_text(
    endpoint_name: str,
    text_chunks: list[str],
    language: str = "ja",
    speaker_id: str = "default",
) -> bytes:
    """
    Synthesize speech using Kotoba TTS bidirectional streaming.

    Args:
        endpoint_name: SageMaker endpoint name
        text_chunks: list of text fragments to stream
        language: target language ("ja" or "en")
        speaker_id: preset voice identifier

    Returns:
        Raw audio bytes (pcm_f32, 24000 Hz, mono)
    """
    # Get AWS credentials
    session = boto3.Session()
    credentials = session.get_credentials().get_frozen_credentials()

    # Configure HTTP/2 client
    config = Config(
        endpoint_uri=f"https://runtime.sagemaker.{region}.amazonaws.com:8443",
        region=region,
        aws_access_key_id=credentials.access_key,
        aws_secret_access_key=credentials.secret_key,
        aws_session_token=credentials.token,
        aws_credentials_identity_resolver=StaticCredentialsResolver(),
        auth_scheme_resolver=HTTPAuthSchemeResolver(),
        auth_schemes={"aws.auth#sigv4": SigV4AuthScheme(service="sagemaker")},
    )

    client = SageMakerRuntimeHTTP2Client(config=config)

    # Start bidirectional stream
    stream = await client.invoke_endpoint_with_bidirectional_stream(
        InvokeEndpointWithBidirectionalStreamInput(
            endpoint_name=endpoint_name,
            model_invocation_path="invocations-bidirectional-stream",
        )
    )

    async def send_event(payload: dict):
        body = json.dumps(payload).encode("utf-8")
        request = RequestStreamEventPayloadPart(
            value=RequestPayloadPart(bytes_=body)
        )
        await stream.input_stream.send(request)

    output = await stream.await_output()
    output_stream = output[1] if isinstance(output, tuple) else output

    # 1) Receive session.created (server sends first)
    message = await output_stream.receive()
    data = json.loads(message.value.bytes_.decode())
    assert data["type"] == "session.created", data
    fmt = data.get("format")
    sr = data.get("sample_rate")
    ch = data.get("channels")
    print(f"Session created: format={fmt}, sr={sr}, ch={ch}")

    # 2) Send open
    await send_event({
        "type": "open",
        "language": language,
        "speaker_id": speaker_id,
    })

    # 3) Start a synthesis turn
    await send_event({"type": "response.create"})
    message = await output_stream.receive()
    data = json.loads(message.value.bytes_.decode())
    assert data["type"] == "response.created", data
    print("Response started")

    audio_buffer = bytearray()

    async def send_text():
        for chunk in text_chunks:
            await send_event({
                "type": "input_text_buffer.append",
                "text": chunk,
            })
        await send_event({"type": "input_text_buffer.commit"})

    async def receive_audio():
        while True:
            message = await output_stream.receive()
            if message is None:
                break
            data = json.loads(message.value.bytes_.decode())
            event_type = data["type"]
            if event_type == "audio.chunk":
                audio_b64 = data.get("audio", "")
                is_final = data.get("isFinal", False)
                audio_buffer.extend(base64.b64decode(audio_b64))
                print(f"Got audio chunk ({len(audio_b64)} b64 chars, isFinal={is_final})")
                if is_final:
                    continue
            elif event_type == "response.done":
                status = data.get("response", {}).get("status")
                print(f"Response done: status={status}")
                if status != "completed":
                    raise RuntimeError(f"Synthesis failed: {data}")
                break
            elif event_type in ("error", "timeout"):
                raise RuntimeError(f"Server event: {data}")

    await asyncio.gather(send_text(), receive_audio())
    await stream.input_stream.close()

    return bytes(audio_buffer)


In [ ]:
# Run synthesis
print("Starting synthesis...\n")

audio_bytes = await synthesize_text(
    endpoint_name=ENDPOINT_NAME,
    text_chunks=TEXT_CHUNKS,
    language=LANGUAGE,
    speaker_id=SPEAKER_ID,
)

print(f"\nReceived {len(audio_bytes)} bytes of pcm_f32 audio")

# Convert pcm_f32 -> int16 and save as WAV for easy playback
samples_f32 = np.frombuffer(audio_bytes, dtype=np.float32)
samples_i16 = np.clip(samples_f32, -1.0, 1.0) * 32767.0
samples_i16 = samples_i16.astype(np.int16)

out_path = "sample_output.wav"
with wave.open(out_path, "wb") as w:
    w.setnchannels(1)
    w.setsampwidth(2)
    w.setframerate(SAMPLE_RATE)
    w.writeframes(samples_i16.tobytes())

print(f"Saved synthesized audio to {out_path} ({len(samples_i16) / SAMPLE_RATE:.2f} s)")

## 5. Clean-up

**Important**: Delete the endpoint when done to avoid ongoing charges.

GPU instances (`ml.g6.2xlarge`) are billed per hour while the endpoint is active.

In [ ]:
# Delete endpoint
print(f"Deleting endpoint '{ENDPOINT_NAME}'...")

sagemaker_session.delete_endpoint(ENDPOINT_NAME)
sagemaker_session.delete_endpoint_config(ENDPOINT_NAME)

print("Endpoint deleted successfully.")

### Unsubscribe from Model Package (Optional)

To unsubscribe from the Kotoba TTS model package:

1. Go to [AWS Marketplace Subscriptions](https://console.aws.amazon.com/marketplace/home#/subscriptions)
2. Find "Kotoba TTS" in your subscriptions
3. Click **Manage** -> **Actions** -> **Cancel subscription**

**Note**: You must delete all endpoints using this model package before unsubscribing.

---

## Appendix A: AWS Marketplace Listing Fields

Reference template for the AWS Marketplace ML Product listing submission.
Fill these when creating or updating the product on the AWS Marketplace Management Portal.

### Required Fields

| Field                        | Sample Value                                                                                       |
| ---------------------------- | -------------------------------------------------------------------------------------------------- |
| Product title                | `Kotoba TTS - Japanese Real-time Speech Synthesis`                                                 |
| Short description (<=160)    | `Real-time Japanese text-to-speech via SageMaker bidirectional streaming. Sub-second latency.`     |
| Highlights (bullet 1)        | `Real-time streaming synthesis with sub-second time-to-first-audio`                                |
| Highlights (bullet 2)        | `Native Japanese voices plus English                         |
| Highlights (bullet 3)        | `GPU-accelerated inference with 24 kHz PCM output`                                                 |
| Categories                   | `Machine Learning` > `Speech` / `Generative AI`                                                    |
| Product logo                 | 1024x1024 PNG (Kotoba brand logo)                                                                  |
| Search keywords              | `tts, japanese, speech synthesis, voice, streaming, realtime, kotoba`                              |
| Supported regions            | `us-east-1, us-west-2, ap-northeast-1` (initial)                                                   |
| Supported instance types     | `ml.g6.2xlarge` (recommended), `ml.g6.12xlarge`, `ml.g5.2xlarge`                                   |
| Input content type           | `application/json` (bidirectional streaming, reference only)                                       |
| Output content type          | `application/json` (`audio.chunk` contains base64 PCM)                                             |
| Max payload size             | N/A (bidirectional streaming, chunked)                                                             |
| Model package ARN            | Generated at build time, e.g. `arn:aws:sagemaker:us-east-1:<account>:model-package/kotoba-tts-ja/1`|
| Sample notebook S3 URI       | `s3://<bucket>/kotoba-tts-ja/kotoba-tts-streaming.ipynb`                                           |
| EULA                         | Kotoba EULA PDF                                                                                    |
| Support contact              | `support@kotoba.tech`                                                                              |
| Support URL                  | `https://kotoba.tech/support`                                                                      |
| Pricing - software           | e.g. `$1.20 / hour` per instance                                                                   |
| Pricing - dimensions         | Per endpoint-hour                                                                                  |
| Refund policy                | `Contact support for refunds within 30 days.`                                                      |
| Versioning                   | SemVer, e.g. `1.0.0` (initial release)                                                             |

### Long Description (English)

```
Kotoba TTS delivers production-grade, low-latency Japanese speech synthesis
for conversational AI, IVR, content localization, and accessibility products.

Key capabilities:
- Real-time bidirectional streaming over SageMaker HTTP/2, with sub-second
  time-to-first-audio.
- Preset Japanese voices and English voices out of the box;
  swap voices per-session by setting `speaker_id`.
- 24 kHz mono PCM output in `pcm_f32` / `pcm16` / `float32` codecs.
- Cancel mid-stream with `response.cancel`; reuse a single session for many
  `response.create` turns.
- GPU-accelerated; tested on `ml.g6.2xlarge` (NVIDIA L4).

Typical use cases: conversational agents, call-center voicebots, audiobook
narration, real-time translation output, and voice-driven UX.

Protocol: SageMaker bidirectional streaming only. Standard POST /invocations
and batch transform are not supported. Requires Python 3.12+ with
aws-sdk-python[sagemaker-runtime-http2].
```

### Long Description (Japanese)

```
Kotoba TTS は、会話 AI・IVR・コンテンツローカライゼーション・アクセシビリティ
向けに、低レイテンシな日本語音声合成を提供するマネージド ML プロダクトです。

主な機能:
- リアルタイム双方向ストリーミング (SageMaker HTTP/2)。
  Time-to-first-audio はサブ秒。
- プリセット日本語話者 (男性・女性) および英語話者をサポート。
  セッション単位で `speaker_id` を指定可能。
- 24 kHz モノラル PCM 出力 (`pcm_f32` / `pcm16` / `float32`)。
- 中断対応: `response.cancel` で生成を即時中断。
  1 セッションで複数ターンの `response.create` を再利用可能。
- GPU 推論: `ml.g6.2xlarge` (NVIDIA L4) で検証済み。

主なユースケース: 会話エージェント、コールセンター音声ボット、オーディオブック
ナレーション、リアルタイム翻訳音声、音声 UX。

プロトコル: SageMaker 双方向ストリーミングのみ対応。POST /invocations および
バッチ変換は非対応。Python 3.12+ と aws-sdk-python[sagemaker-runtime-http2]
が必要です。
```

---

## Support

For technical support, please contact Kotoba AI support.

## References

- [SageMaker Bidirectional Streaming Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints-test-endpoints.html)
- [AWS SDK for Python (Boto3)](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html)